# Lab 2 — Sparse vs dense vs hybrid (EP10)

From Lab 1: a long question with the book's title pushed the "library card" chunk down to rank 8, and the answer that was in our library was lost.
In this lab we try the other retrieval methods of §16.3 on the same library

**What you'll do**
1. Rebuild Lab 1's library (already written for you)
2. Write **BM25** ourselves, following Equation 16.4
3. See when BM25 wins and when dense wins
4. Write **RRF** (Equation 16.8), check it against the book's numbers, and merge the two into **hybrid**
5. Measure with 10 questions whose answer pages we know: **hit@3**

This lab never calls Gemma — only the embedding model

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.3 — CC BY-SA 4.0

## 0. Setup

The same as Lab 1: Python 3.10+ or Colab, and `OPENROUTER_API_KEY` (Colab: 🔑 Secrets · your own machine: a `.env` file).
This lab only uses the embedding model — it costs about 0.05 US cents.
How to set up the key, and why we chose these models, is in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3"

## 1. Lab 1's library

The next cell is Lab 1's parts 3–5 condensed: load textbook v1, pages 292–374 → 512/64 chunks → embeddings

In [ ]:
import os, re, math, urllib.request
from collections import Counter
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
tok = Tokenizer.from_pretrained("google/gemma-4-26b-a4b-it")

urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = [{"source": "hitchhikers-guide-v1.pdf", "page": n,
          "text": reader.pages[n - 1].extract_text().replace(HEADER, "")}
         for n in range(292, 375)]

def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    return [{"source": page["source"], "page": page["page"],
             "text": tok.decode(ids[start:start + W])}
            for start in range(0, max(len(ids) - O, 1), W - O)]

def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64], extra_body=EMBED_EXTRA)
        vectors += [d.embedding for d in r.data]
    V = np.array(vectors)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

chunks = [c for p in pages for c in chunk_page(p)]
E = embed([c["text"] for c in chunks])
print(len(chunks), "chunks,", E.shape)

In [ ]:
def dense_rank(query):
    scores = E @ embed([query])[0]
    return list(np.argsort(-scores))

## 2. BM25 step 1: library statistics

$$\text{BM25}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d)+k_1\left(1-b+b\,\frac{|d|}{\text{avgdl}}\right)}\qquad(16.4)$$

Before scoring we need: the number of chunks `N` · their average length `avgdl` · how many chunks contain each word, `df`.
The book writes `IDF(t)` without a formula — we use the form BM25 usually uses: $\ln\left(\frac{N-n+0.5}{n+0.5}+1\right)$, where `n` = the number of chunks containing the word

In [ ]:
def words(text):
    return re.findall(r"\w+", text.lower())

docs = [words(c["text"]) for c in chunks]
N = len(docs)
avgdl = sum(len(d) for d in docs) / N
df = Counter(w for d in docs for w in set(d))
print(N, "chunks · avgdl =", round(avgdl), "words")
print("df['rag'] =", df["rag"], "· df['plaid'] =", df["plaid"])

In [ ]:
def idf(t):
    n = df.get(t, 0)
    return math.log((N - n + 0.5) / (n + 0.5) + 1)
print("idf('rag') =", round(idf("rag"), 2), "· idf('plaid') =", round(idf("plaid"), 2))

## 3. BM25 step 2: score every chunk

The three ideas from the theory video: rare words count more (`IDF`) · a repeated word adds, but levels off (`k1`) · long chunks lose a little (`b`).
The book: $k_1 \in [1.2, 2.0]$, $b = 0.75$ — we use $k_1 = 1.5$

In [ ]:
def bm25_rank(query, k1=1.5, b=0.75):
    terms = words(query)
    scores = []
    for d in docs:
        tf = Counter(d)
        s = 0.0
        for t in terms:
            f = tf[t]
            s += idf(t) * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
        scores.append(s)
    return list(np.argsort(-np.array(scores)))

def show(name, ranked, k=3):
    print(f"{name:7s}", [chunks[i]["page"] for i in ranked[:k]])

## 4. Who wins when

The book: sparse still wins on **exact matches** (codes, proper nouns, rare terms) · dense wins on **meaning**, when the question uses different words from the document

In [ ]:
q = "What is PLAID?"
show("bm25", bm25_rank(q))
show("dense", dense_rank(q))

In [ ]:
q = "Which chunks should I give the model if small ones retrieve better but lack context?"
show("bm25", bm25_rank(q))
show("dense", dense_rank(q))

## 5. Hybrid with RRF

$$\text{RRF}(d)=\sum_{r\in R}\frac{1}{k+\text{rank}_r(d)},\quad k=60\qquad(16.8)$$

It uses **ranks** instead of scores, because BM25 and cosine scores are on different scales — the book's example: ranks 3 and 7 → 0.0308 · rank 1 in both → 0.0328.
This function is `reciprocal_rank_fusion` from the book's Listing 16.8

In [ ]:
def rrf(ranked_lists, k=60):
    scores = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

bm = ["x", "y", "d", "z"]                       # d at rank 3
de = ["p", "q", "r", "s", "t", "u", "d"]        # d at rank 7
print(round(1 / (60 + 3) + 1 / (60 + 7), 4), round(2 / 61, 4))
print(rrf([bm, de])[:3])

In [ ]:
def hybrid_rank(query):
    return rrf([bm25_rank(query), dense_rank(query)])

q = "When should I fine-tune a model instead of looking things up?"
for name, fn in [("bm25", bm25_rank), ("dense", dense_rank), ("hybrid", hybrid_rank)]:
    show(name, fn(q))

## 6. Measure: hit@3 on 10 questions

For each question we know the answer page (`gold`) · `first_hit` = the first rank that gives a chunk from a gold page · **hit@3** = whether it's in the top three.
There are rare-term questions (PLAID, HyDE, FAISS) and questions in our own words

In [ ]:
TESTS = [
    ("What is PLAID?", {301}),
    ("What is HyDE?", {303}),
    ("What does FAISS stand for?", {298}),
    ("How can I search millions of vectors quickly?", {298}),
    ("Which method compares every query token with every document token?", {300}),
    ("How do I make keyword search understand synonyms?", {299}),
    ("When should I fine-tune a model instead of looking things up?", {296}),
    ("Which chunks should I give the model if small ones retrieve better but lack context?", {302, 303}),
    ("How do I combine keyword search and embedding search?", {298}),
    ("What analogy does the textbook use to explain RAG?", {295}),
]

def first_hit(ranked, gold):
    for rank, i in enumerate(ranked, start=1):
        if chunks[i]["page"] in gold:
            return rank

print(f"{'question':58s} bm25 dense hybrid")
hits = Counter()
for q, gold in TESTS:
    row = []
    for name, fn in [("bm25", bm25_rank), ("dense", dense_rank), ("hybrid", hybrid_rank)]:
        r = first_hit(fn(q), gold)
        hits[name] += r <= 3
        row.append(r)
    print(f"{q[:56]:58s} {row[0]:4d} {row[1]:5d} {row[2]:6d}")
print("hit@3:", dict(hits), "of", len(TESTS))

## 7. Lab 1's missed question

The long question with the book's title — does hybrid help?

In [ ]:
question = ("In Roitman's book The Hitchhiker's Guide to Agentic AI, "
            "what analogy does the RAG chapter use to explain RAG?")
for name, fn in [("bm25", bm25_rank), ("dense", dense_rank), ("hybrid", hybrid_rank)]:
    print(name, first_hit(fn(question), {295}))

## Summary

| | What we saw in the lab |
|---|---|
| IDF | `rag` in 34 chunks → ≈ 1.4 · `plaid` in 1 chunk → ≈ 4.5 |
| BM25 wins | "What is PLAID?" → page 301 at rank 1 · dense misses it |
| Dense wins | parent-child asked without the term → pages 302–303 on top · BM25 misses them |
| RRF | ranks 3 + 7 → 0.0308 · 1 + 1 → 0.0328, matching the book |
| Hybrid | fine-tune vs look up: in neither list's top three, but hybrid ranks it 1st |
| hit@3 (10 questions) | BM25 4 · dense 4 · **hybrid 8** — but PLAID and the analogy question got worse |
| Lab 1's missed question | BM25 10 · dense 8 · hybrid 5 — still missed |

**Exercises**
1. Try `k1 = 1.2` and `k1 = 2.0` (the book's range) — does BM25's hit@3 change?
2. Try `rrf(…, k=1)` instead of 60 — the top ranks gain more weight; what happens to hybrid's hit@3?
3. Write 5 questions of your own with their answer pages, and measure again

**Next — Lab 3:** chunk the same library several ways and measure hit@3 the same way